# Preprocess hagrid_small parquet -> train-ready CSV

This notebook reads all parquet files under `hagrid_small/data`, runs MediaPipe preprocessing, and saves outputs in a train-compatible table format.

In [1]:
import sys
!{sys.executable} -m pip install -U datasets mediapipe pillow tqdm pandas numpy

  Using cached numpy-2.4.6-cp311-cp311-macosx_14_0_x86_64.whl.metadata (6.6 kB)


In [2]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

from hand_preprocess import MediaPipeHandPreprocessor

PROJECT_DIR = Path.cwd()
PARQUET_GLOB = str(PROJECT_DIR / "hagrid_small" / "data" / "*.parquet")
OUT_DIR = PROJECT_DIR / "processed_hagrid_small"
CROP_DIR = OUT_DIR / "crops"
LM_DIR = OUT_DIR / "landmarks"
CSV_PATH = OUT_DIR / "labels_fixed.csv"

for p in [OUT_DIR, CROP_DIR, LM_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("PARQUET_GLOB:", PARQUET_GLOB)
print("OUTPUT CSV:", CSV_PATH)

PROJECT_DIR: /Users/chenyingda/Documents/2026nthu/Multimedia/final_project/HandGesture_project
PARQUET_GLOB: /Users/chenyingda/Documents/2026nthu/Multimedia/final_project/HandGesture_project/hagrid_small/data/*.parquet
OUTPUT CSV: /Users/chenyingda/Documents/2026nthu/Multimedia/final_project/HandGesture_project/processed_hagrid_small/labels_fixed.csv


In [3]:
# Label mapping to match train.ipynb classes
# 0: N_A, 1: fist, 2: like, 3: ok, 4: one, 5: palm
TARGET_MAP = {"fist": 1, "like": 2, "ok": 3, "one": 4, "palm": 5}
ID_TO_NAME = {0: "N_A", 1: "fist", 2: "like", 3: "ok", 4: "one", 5: "palm"}

FALLBACK_LABEL_NAMES = [
    "call", "dislike", "fist", "four", "like", "mute",
    "no_gesture", "ok", "one", "palm", "peace", "peace_inverted",
    "rock", "stop", "stop_inverted", "three", "three2", "two_up", "two_up_inverted",
]

In [4]:
ds = load_dataset(
    "parquet",
    data_files=PARQUET_GLOB,
    split="train",
)

label_names = None
if "label" in ds.features and hasattr(ds.features["label"], "names"):
    label_names = ds.features["label"].names
if not label_names:
    label_names = FALLBACK_LABEL_NAMES

print(f"Loaded dataset: {len(ds)} samples")
print("First 10 label names:", label_names[:10])

Loaded dataset: 153735 samples
First 10 label names: ['call', 'dislike', 'fist', 'four', 'like', 'mute', 'no_gesture', 'ok', 'one', 'palm']


In [5]:
def get_label_name(label_id: int) -> str:
    if 0 <= label_id < len(label_names):
        return str(label_names[label_id])
    return f"unknown_{label_id}"

def load_pil_image(image_field):
    if isinstance(image_field, Image.Image):
        return image_field.convert("RGB")

    if isinstance(image_field, dict):
        img_bytes = image_field.get("bytes", None)
        if img_bytes is not None:
            return Image.open(io.BytesIO(img_bytes)).convert("RGB")

        img_path = image_field.get("path", None)
        if img_path:
            return Image.open(img_path).convert("RGB")

    raise ValueError("Unsupported image field format")

In [6]:
MAX_SAMPLES = None  # set an integer for quick test, e.g. 1000
iter_ds = ds if MAX_SAMPLES is None else ds.select(range(min(MAX_SAMPLES, len(ds))))

records = []
fail_count = 0

with MediaPipeHandPreprocessor() as preprocessor:
    for idx, sample in enumerate(tqdm(iter_ds, desc="Preprocessing")):
        try:
            img = load_pil_image(sample["image"])
        except Exception:
            fail_count += 1
            continue

        result = preprocessor.preprocess_image(img)
        if result is None:
            fail_count += 1
            continue

        crop, landmarks = result

        src_label = int(sample["label"])
        src_name = get_label_name(src_label)

        label = TARGET_MAP.get(src_name, 0)
        label_name = ID_TO_NAME[label]

        safe_name = src_name.replace("/", "_").replace(" ", "_")
        base_name = f"{idx:08d}_{safe_name}"

        crop_path = (CROP_DIR / f"{base_name}.png").resolve()
        lm_path = (LM_DIR / f"{base_name}.npy").resolve()

        Image.fromarray(crop).save(crop_path)
        np.save(lm_path, landmarks.astype(np.float32))

        records.append({
            "idx": idx,
            "crop_path": str(crop_path),
            "landmark_path": str(lm_path),
            "label": label,
            "label_name": label_name,
            "source_label": src_label,
            "source_label_name": src_name,
        })

df = pd.DataFrame(records)
df.to_csv(CSV_PATH, index=False)

print("Saved rows:", len(df))
print("Failed/Skipped:", fail_count)
print("CSV:", CSV_PATH)
print("\nClass distribution:")
print(df["label_name"].value_counts())

I0000 00:00:1780080902.846270 54880696 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M1
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1780080902.912271 54889420 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780080902.946493 54889426 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Preprocessing:   0%|          | 0/153735 [00:00<?, ?it/s]

W0000 00:00:1780080903.057799 54889420 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Saved rows: 140938
Failed/Skipped: 12797
CSV: /Users/chenyingda/Documents/2026nthu/Multimedia/final_project/HandGesture_project/processed_hagrid_small/labels_fixed.csv

Class distribution:
label_name
N_A     108010
palm      6878
one       6690
ok        6665
like      6381
fist      6314
Name: count, dtype: int64


In [7]:
import numpy as np

df_check = pd.read_csv(CSV_PATH)
print(df_check.head())
print("total:", len(df_check))
print(df_check["label"].value_counts().sort_index())

if len(df_check) > 0:
    lm = np.load(df_check.iloc[0]["landmark_path"])
    print("landmark shape:", lm.shape)

   idx                                          crop_path  \
0    0  /Users/chenyingda/Documents/2026nthu/Multimedi...   
1    1  /Users/chenyingda/Documents/2026nthu/Multimedi...   
2    2  /Users/chenyingda/Documents/2026nthu/Multimedi...   
3    3  /Users/chenyingda/Documents/2026nthu/Multimedi...   
4    4  /Users/chenyingda/Documents/2026nthu/Multimedi...   

                                       landmark_path  label label_name  \
0  /Users/chenyingda/Documents/2026nthu/Multimedi...      0        N_A   
1  /Users/chenyingda/Documents/2026nthu/Multimedi...      0        N_A   
2  /Users/chenyingda/Documents/2026nthu/Multimedi...      0        N_A   
3  /Users/chenyingda/Documents/2026nthu/Multimedi...      0        N_A   
4  /Users/chenyingda/Documents/2026nthu/Multimedi...      0        N_A   

   source_label source_label_name  
0             0              call  
1             0              call  
2             0              call  
3             0              call  
4       

## Use in train.ipynb

Replace CSV path with:

`df = pd.read_csv("processed_hagrid_small/labels_fixed.csv")`